In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from PIL import Image, ImageChops

from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, real_amplitudes


# ============================================================
# Standalone illustration settings
# ============================================================
NUM_QUBITS = 2
FEATURE_MAP_REPS = 1
ANSATZ_REPS = 2

out_dir = Path(".")
circuit_png = out_dir / "illustration_actual_vqc_circuit.png"
diagram_png = out_dir / "noise_to_vqc_output_diagram.png"
diagram_svg = out_dir / "noise_to_vqc_output_diagram.svg"


# ============================================================
# Build standalone illustrative VQC circuit
# ============================================================
feature_map = zz_feature_map(
    feature_dimension=NUM_QUBITS,
    reps=FEATURE_MAP_REPS,
)

ansatz = real_amplitudes(
    num_qubits=NUM_QUBITS,
    reps=ANSATZ_REPS,
)

vqc_circuit = QuantumCircuit(NUM_QUBITS, name="VQC")
vqc_circuit.compose(feature_map, inplace=True)
vqc_circuit.compose(ansatz, inplace=True)

# For the illustration, measurement is useful because the slide is about output changes.
vqc_circuit.measure_all()

# Draw the actual Qiskit circuit as an image.
circuit_fig = vqc_circuit.decompose().draw(
    output="mpl",
    fold=-1,
    style={
        "backgroundcolor": "#ffffff",
        "fontsize": 10,
        "subfontsize": 8,
        "displaytext": {
            "rx": "Rx",
            "ry": "Ry",
            "rz": "Rz",
            "cx": "CX",
        },
    },
)

circuit_fig.savefig(
    circuit_png,
    bbox_inches="tight",
    facecolor="white",
    dpi=800,
)

plt.close(circuit_fig)


# ============================================================
# Trim white margins from circuit image
# ============================================================
def trim_white_margin(image):
    image = image.convert("RGB")
    background = Image.new("RGB", image.size, (255, 255, 255))
    diff = ImageChops.difference(image, background)
    bbox = diff.getbbox()

    if bbox is None:
        return image

    return image.crop(bbox)


circuit_img = trim_white_margin(Image.open(circuit_png))


# ============================================================
# Main figure setup
# ============================================================
fig, ax = plt.subplots(figsize=(12, 4.1), dpi=800)

ax.set_xlim(0, 12)
ax.set_ylim(0, 4.1)
ax.axis("off")


# ============================================================
# Helper functions
# ============================================================
def rounded_box(x, y, w, h, edgecolor, facecolor, linewidth=2.0):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.035,rounding_size=0.11",
        linewidth=linewidth,
        edgecolor=edgecolor,
        facecolor=facecolor,
    )
    ax.add_patch(box)
    return box


def add_arrow(x1, y1, x2, y2):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            arrowstyle="simple",
            mutation_scale=18,
            linewidth=0,
            color="#222222",
        )
    )


# ============================================================
# Left block: Noise
# ============================================================
rounded_box(
    x=0.70,
    y=1.12,
    w=2.05,
    h=2.08,
    edgecolor="#b91c1c",
    facecolor="#fff7f7",
)

ax.text(
    1.725,
    2.72,
    "Noise",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color="#b91c1c",
)

ax.text(
    1.725,
    2.42,
    "simulated quantum noise",
    ha="center",
    va="center",
    fontsize=9,
    color="#7f1d1d",
)

# simple visual icon only
rng = np.random.default_rng(42)
x_noise = rng.uniform(1.12, 2.32, 8)
y_noise = rng.uniform(1.58, 1.92, 8)

ax.scatter(
    x_noise,
    y_noise,
    s=24,
    color="#ef4444",
    alpha=0.65,
    edgecolors="none",
)

for x0, y0 in [(1.10, 1.73), (2.05, 1.73)]:
    xs = np.linspace(0, 0.32, 80) + x0
    ys = 0.035 * np.sin(np.linspace(0, 4 * np.pi, 80)) + y0
    ax.plot(xs, ys, color="#dc2626", linewidth=1.5)
# ============================================================
# Center block: Qiskit-rendered VQC circuit
# ============================================================
rounded_box(
    x=3.50,
    y=0.70,
    w=5.05,
    h=2.95,
    edgecolor="#2563eb",
    facecolor="#f8fbff",
)

ax.text(
    6.025,
    3.33,
    "Two-Qubit VQC",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color="#1d4ed8",
)

ax.text(
    6.025,
    3.07,
    "ZZFeatureMap + RealAmplitudes",
    ha="center",
    va="center",
    fontsize=9.5,
    color="#1e40af",
)

# Insert real Qiskit circuit image.
# The extent controls how large the circuit appears inside the blue box.
ax.imshow(
    circuit_img,
    extent=[3.78, 8.28, 1.18, 2.88],
    aspect="auto",
    zorder=2,
)

ax.text(
    6.025,
    0.96,
    "fixed trained parameters; noise changes the measured output",
    ha="center",
    va="center",
    fontsize=8.5,
    color="#374151",
)


# ============================================================
# Right block: Output changes
# ============================================================
rounded_box(
    x=9.45,
    y=1.05,
    w=2.05,
    h=2.18,
    edgecolor="#166534",
    facecolor="#f6fff8",
)

ax.text(
    10.475,
    2.82,
    "Output",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color="#166534",
)

ax.text(
    10.475,
    2.54,
    "changes",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color="#166534",
)

# Output bars
baseline_y = 1.50

ax.plot(
    [9.92, 11.06],
    [baseline_y, baseline_y],
    color="#111111",
    linewidth=1.2,
)

bar_x = np.array([10.05, 10.32, 10.59, 10.86])
before = np.array([0.72, 0.40, 0.66, 0.36])
after = np.array([0.34, 0.78, 0.28, 0.58])

for x, h in zip(bar_x, before):
    ax.add_patch(
        Rectangle(
            (x - 0.045, baseline_y),
            0.09,
            h,
            linewidth=1.1,
            edgecolor="#15803d",
            facecolor="none",
            linestyle="--",
        )
    )

for x, h in zip(bar_x + 0.08, after):
    ax.add_patch(
        Rectangle(
            (x - 0.045, baseline_y),
            0.09,
            h,
            linewidth=1.0,
            edgecolor="#15803d",
            facecolor="#84cc16",
            alpha=0.85,
        )
    )

# shorter labels so they stay inside the box
ax.text(
    10.475,
    1.26,
    "dashed: before",
    ha="center",
    va="center",
    fontsize=7.6,
    color="#374151",
)

ax.text(
    10.475,
    1.11,
    "solid: after",
    ha="center",
    va="center",
    fontsize=7.6,
    color="#374151",
)


# ============================================================
# Arrows
# ============================================================
add_arrow(2.95, 2.16, 3.32, 2.16)
add_arrow(8.72, 2.16, 9.25, 2.16)


# ============================================================
# Caption
# ============================================================
ax.text(
    6.0,
    0.30,
    "Noise perturbs the measured circuit behavior, so the VQC output distribution can shift.",
    ha="center",
    va="center",
    fontsize=10,
    color="#374151",
)


# ============================================================
# Save and show
# ============================================================
fig.tight_layout()

fig.savefig(
    diagram_png,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    diagram_svg,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

print(f"Saved circuit image to: {circuit_png.resolve()}")
print(f"Saved diagram PNG to:  {diagram_png.resolve()}")
print(f"Saved diagram SVG to:  {diagram_svg.resolve()}")

In [ ]:
from pathlib import Path
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter


# ============================================
# Output files
# ============================================
out_dir = Path(".")
png_path = out_dir / "text_tfidf_pca_vqc_flow_qiskit.png"
svg_path = out_dir / "text_tfidf_pca_vqc_flow_qiskit.svg"


# ============================================
# Build a proper 2-qubit VQC circuit in Qiskit
# ============================================
def build_two_qubit_vqc():
    x0 = Parameter("x0")
    x1 = Parameter("x1")
    t0 = Parameter("θ0")
    t1 = Parameter("θ1")
    t2 = Parameter("θ2")
    t3 = Parameter("θ3")

    qc = QuantumCircuit(2, name="Two-Qubit VQC")

    # simple feature encoding
    qc.h(0)
    qc.h(1)
    qc.rz(x0, 0)
    qc.rz(x1, 1)
    qc.cx(0, 1)

    # variational layer
    qc.ry(t0, 0)
    qc.ry(t1, 1)
    qc.cx(0, 1)
    qc.ry(t2, 0)
    qc.ry(t3, 1)

    return qc


def render_qiskit_circuit_image():
    qc = build_two_qubit_vqc()

    qfig = qc.draw(
        output="mpl",
        fold=-1,
        style={
            "fontsize": 10,
            "subfontsize": 8,
        },
        idle_wires=False,
    )

    buf = BytesIO()
    qfig.savefig(buf, format="png", dpi=800, bbox_inches="tight", facecolor="white")
    plt.close(qfig)

    buf.seek(0)
    img = mpimg.imread(buf)
    buf.close()
    return img


circuit_img = render_qiskit_circuit_image()


# ============================================
# Main figure
# ============================================
fig, ax = plt.subplots(figsize=(14, 3.6), dpi=800)
ax.set_xlim(0, 14)
ax.set_ylim(0, 3.6)
ax.axis("off")


# ============================================
# Helpers
# ============================================
def rounded_box(x, y, w, h, edgecolor, facecolor, lw=2.2):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.045,rounding_size=0.12",
        linewidth=lw,
        edgecolor=edgecolor,
        facecolor=facecolor,
    )
    ax.add_patch(box)
    return box


def label(x, y, text, color, fontsize=14):
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold",
        color=color,
    )


def sublabel(x, y, text, color, fontsize=8.5):
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        color=color,
    )


def arrow(x1, y1, x2, y2):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            arrowstyle="simple",
            mutation_scale=18,
            linewidth=0,
            color="#222222",
        )
    )


# ============================================
# Layout
# ============================================
box_y = 0.78
box_h = 1.95
small_w = 2.2
vqc_w = 3.3

x_text = 0.45
x_tfidf = 3.35
x_pca = 6.25
x_vqc = 9.15

mid_y = box_y + box_h / 2


# ============================================
# Box 1: Text Data
# ============================================
rounded_box(
    x_text, box_y, small_w, box_h,
    edgecolor="#b91c1c",
    facecolor="#fff7f7",
)

label(x_text + small_w / 2, 2.37, "Text Data", "#b91c1c")
sublabel(x_text + small_w / 2, 2.09, "raw sentences", "#7f1d1d")

doc_x = x_text + 0.58
doc_y = 1.13
doc_w = 1.02
doc_h = 0.70

ax.add_patch(
    Rectangle(
        (doc_x, doc_y),
        doc_w,
        doc_h,
        linewidth=1.4,
        edgecolor="#b91c1c",
        facecolor="white",
    )
)

for i, width in enumerate([0.72, 0.56, 0.80, 0.47]):
    ax.plot(
        [doc_x + 0.14, doc_x + 0.14 + width],
        [doc_y + 0.54 - i * 0.12, doc_y + 0.54 - i * 0.12],
        color="#ef4444",
        linewidth=1.4,
    )


# ============================================
# Box 2: TF-IDF
# ============================================
rounded_box(
    x_tfidf, box_y, small_w, box_h,
    edgecolor="#7c3aed",
    facecolor="#f5f3ff",
)

label(x_tfidf + small_w / 2, 2.37, "TF-IDF", "#6d28d9")
sublabel(x_tfidf + small_w / 2, 2.09, "term weights", "#4c1d95")

base_x = x_tfidf + 0.58
base_y = 1.12
bar_heights = [0.34, 0.62, 0.44, 0.78, 0.52]

for i, h in enumerate(bar_heights):
    ax.add_patch(
        Rectangle(
            (base_x + i * 0.23, base_y),
            0.13,
            h,
            linewidth=1.0,
            edgecolor="#6d28d9",
            facecolor="#a78bfa",
            alpha=0.9,
        )
    )

ax.plot(
    [base_x - 0.06, base_x + 1.03],
    [base_y, base_y],
    color="#4c1d95",
    linewidth=1.2,
)


# ============================================
# Box 3: PCA(2)
# ============================================
rounded_box(
    x_pca, box_y, small_w, box_h,
    edgecolor="#0369a1",
    facecolor="#f0f9ff",
)

label(x_pca + small_w / 2, 2.37, "PCA(2)", "#0369a1")
sublabel(x_pca + small_w / 2, 2.09, "2 features", "#075985")

origin_x = x_pca + 0.62
origin_y = 1.13

ax.plot([origin_x, origin_x + 1.02], [origin_y, origin_y], color="#075985", linewidth=1.2)
ax.plot([origin_x, origin_x], [origin_y, origin_y + 0.78], color="#075985", linewidth=1.2)

pts = np.array([
    [0.18, 0.22],
    [0.33, 0.48],
    [0.52, 0.34],
    [0.72, 0.62],
    [0.86, 0.28],
])

ax.scatter(
    origin_x + pts[:, 0],
    origin_y + pts[:, 1],
    s=30,
    color="#38bdf8",
    edgecolors="#0369a1",
    linewidths=0.8,
)


# ============================================
# Box 4: Two-Qubit VQC Input
# ============================================
rounded_box(
    x_vqc, box_y, vqc_w, box_h,
    edgecolor="#166534",
    facecolor="#f0fdf4",
)

label(x_vqc + vqc_w / 2, 2.37, "Two-Qubit VQC Input", "#166534", fontsize=13)
sublabel(
    x_vqc + vqc_w / 2,
    2.09,
    "proper Qiskit-drawn circuit",
    "#14532d",
    fontsize=8.5,
)

# embed the qiskit circuit image inside the box
img_left = x_vqc + 0.22
img_right = x_vqc + vqc_w - 0.22
img_bottom = 1.06
img_top = 1.93

ax.imshow(
    circuit_img,
    extent=[img_left, img_right, img_bottom, img_top],
    aspect="auto",
    zorder=3,
)

# little caption under the circuit
ax.text(
    x_vqc + vqc_w / 2,
    0.98,
    "2 PCA components encoded into a 2-qubit VQC",
    ha="center",
    va="center",
    fontsize=8.2,
    color="#14532d",
)


# ============================================
# Arrows
# ============================================
arrow(x_text + small_w + 0.18, mid_y, x_tfidf - 0.18, mid_y)
arrow(x_tfidf + small_w + 0.18, mid_y, x_pca - 0.18, mid_y)
arrow(x_pca + small_w + 0.18, mid_y, x_vqc - 0.18, mid_y)


# ============================================
# Caption
# ============================================
ax.text(
    7.0,
    0.28,
    "Text is converted to TF-IDF features, reduced to two PCA components, and used as input to a two-qubit VQC.",
    ha="center",
    va="center",
    fontsize=9.5,
    color="#374151",
)


# ============================================
# Save
# ============================================
fig.tight_layout()
fig.savefig(png_path, bbox_inches="tight", facecolor="white")
fig.savefig(svg_path, bbox_inches="tight", facecolor="white")

plt.show()

print(f"Saved PNG to: {png_path.resolve()}")
print(f"Saved SVG to: {svg_path.resolve()}")

In [ ]:
from pathlib import Path
from io import BytesIO

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Circle

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter


# ============================================
# Output files
# ============================================
out_dir = Path(".")
png_path = out_dir / "featuremap_ansatz_measurement_classlabel_flow.png"
svg_path = out_dir / "featuremap_ansatz_measurement_classlabel_flow.svg"


# ============================================
# Qiskit helpers
# ============================================
def render_circuit_image(qc, dpi=800):
    fig = qc.draw(
        output="mpl",
        fold=-1,
        idle_wires=False,
        style={
            "fontsize": 9,
            "subfontsize": 7,
        },
    )

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    img = mpimg.imread(buf)
    buf.close()
    return img


def build_feature_map_circuit():
    x0 = Parameter("x0")
    x1 = Parameter("x1")

    qc = QuantumCircuit(2, name="Feature Map")
    qc.h(0)
    qc.h(1)
    qc.rz(x0, 0)
    qc.rz(x1, 1)
    qc.cx(0, 1)
    return qc


def build_ansatz_circuit():
    t0 = Parameter("θ0")
    t1 = Parameter("θ1")
    t2 = Parameter("θ2")
    t3 = Parameter("θ3")

    qc = QuantumCircuit(2, name="Ansatz")
    qc.ry(t0, 0)
    qc.ry(t1, 1)
    qc.cx(0, 1)
    qc.ry(t2, 0)
    qc.ry(t3, 1)
    return qc


def build_measurement_circuit():
    qc = QuantumCircuit(2, 2, name="Measurement")
    qc.measure(0, 0)
    qc.measure(1, 1)
    return qc


feature_map_img = render_circuit_image(build_feature_map_circuit())
ansatz_img = render_circuit_image(build_ansatz_circuit())
measurement_img = render_circuit_image(build_measurement_circuit())


# ============================================
# Main figure
# ============================================
fig, ax = plt.subplots(figsize=(14.5, 3.8), dpi=800)
ax.set_xlim(0, 14.5)
ax.set_ylim(0, 3.8)
ax.axis("off")


# ============================================
# Drawing helpers
# ============================================
def rounded_box(x, y, w, h, edgecolor, facecolor, linewidth=2.2):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.045,rounding_size=0.12",
        linewidth=linewidth,
        edgecolor=edgecolor,
        facecolor=facecolor,
    )
    ax.add_patch(box)
    return box


def label(x, y, text, color, fontsize=13.5):
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold",
        color=color,
    )


def sublabel(x, y, text, color, fontsize=8.3):
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        color=color,
    )


def arrow(x1, y1, x2, y2):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            arrowstyle="simple",
            mutation_scale=18,
            linewidth=0,
            color="#222222",
        )
    )


# ============================================
# Layout
# ============================================
box_y = 0.78
box_h = 2.10
small_w = 2.55
class_w = 2.85

x1 = 0.45
x2 = 3.55
x3 = 6.65
x4 = 9.75

mid_y = box_y + box_h / 2


# ============================================
# Box 1: Feature Map
# ============================================
rounded_box(
    x1, box_y, small_w, box_h,
    edgecolor="#b91c1c",
    facecolor="#fff7f7",
)

label(x1 + small_w / 2, 2.48, "Feature Map", "#b91c1c")
sublabel(x1 + small_w / 2, 2.18, "encode input features", "#7f1d1d")

ax.imshow(
    feature_map_img,
    extent=[x1 + 0.18, x1 + small_w - 0.18, 1.08, 1.95],
    aspect="auto",
    zorder=3,
)


# ============================================
# Box 2: Ansatz
# ============================================
rounded_box(
    x2, box_y, small_w, box_h,
    edgecolor="#7c3aed",
    facecolor="#f5f3ff",
)

label(x2 + small_w / 2, 2.48, "Ansatz", "#6d28d9")
sublabel(x2 + small_w / 2, 2.18, "trainable variational layer", "#4c1d95")

ax.imshow(
    ansatz_img,
    extent=[x2 + 0.18, x2 + small_w - 0.18, 1.08, 1.95],
    aspect="auto",
    zorder=3,
)


# ============================================
# Box 3: Measurement
# ============================================
rounded_box(
    x3, box_y, small_w, box_h,
    edgecolor="#0369a1",
    facecolor="#f0f9ff",
)

label(x3 + small_w / 2, 2.48, "Measurement", "#0369a1")
sublabel(x3 + small_w / 2, 2.18, "readout from the circuit", "#075985")

ax.imshow(
    measurement_img,
    extent=[x3 + 0.40, x3 + small_w - 0.40, 1.16, 1.88],
    aspect="auto",
    zorder=3,
)

# small classical-bit cue
ax.text(
    x3 + small_w / 2,
    1.00,
    "bitstring / counts",
    ha="center",
    va="center",
    fontsize=8.0,
    color="#075985",
)


# ============================================
# Box 4: Class Label
# ============================================
rounded_box(
    x4, box_y, class_w, box_h,
    edgecolor="#166534",
    facecolor="#f0fdf4",
)

# Center reference for the whole box
box_center_x = x4 + class_w / 2

label(box_center_x, 2.48, "Class Label", "#166534")
sublabel(box_center_x, 2.18, "final prediction", "#14532d")

# Center reference for the icon row
center_x = box_center_x
center_y = 1.50

# The whole icon group spans evenly around center_x:
# left edge = center_x - 1.05
# right edge = center_x + 1.05

# left badge
left_badge_x = center_x - 1.05
left_badge_w = 0.65

ax.add_patch(
    FancyBboxPatch(
        (left_badge_x, center_y - 0.22),
        left_badge_w,
        0.44,
        boxstyle="round,pad=0.02,rounding_size=0.08",
        linewidth=1.2,
        edgecolor="#166534",
        facecolor="white",
    )
)

ax.text(
    left_badge_x + left_badge_w / 2,
    center_y,
    "0",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color="#166534",
)

# arrow between badges
ax.add_patch(
    FancyArrowPatch(
        (center_x - 0.20, center_y),
        (center_x + 0.10, center_y),
        arrowstyle="->",
        mutation_scale=14,
        linewidth=1.6,
        color="#166534",
    )
)

# right highlighted badge
right_badge_x = center_x + 0.20
right_badge_w = 0.85

ax.add_patch(
    FancyBboxPatch(
        (right_badge_x, center_y - 0.27),
        right_badge_w,
        0.54,
        boxstyle="round,pad=0.03,rounding_size=0.10",
        linewidth=1.5,
        edgecolor="#166534",
        facecolor="#bbf7d0",
    )
)

ax.text(
    right_badge_x + right_badge_w / 2,
    center_y,
    "Label",
    ha="center",
    va="center",
    fontsize=9.8,
    fontweight="bold",
    color="#166534",
)

ax.text(
    box_center_x,
    1.03,
    "binary decision output",
    ha="center",
    va="center",
    fontsize=8.0,
    color="#14532d",
)

# ============================================
# Arrows
# ============================================
arrow(x1 + small_w + 0.16, mid_y, x2 - 0.16, mid_y)
arrow(x2 + small_w + 0.16, mid_y, x3 - 0.16, mid_y)
arrow(x3 + small_w + 0.16, mid_y, x4 - 0.16, mid_y)


# ============================================
# Caption
# ============================================
ax.text(
    7.25,
    0.28,
    "The feature map encodes the input, the ansatz applies trainable operations, measurement produces readout values, and the final output is mapped to a class label.",
    ha="center",
    va="center",
    fontsize=9.2,
    color="#374151",
)


# ============================================
# Save and show
# ============================================
fig.tight_layout()

fig.savefig(png_path, bbox_inches="tight", facecolor="white")
fig.savefig(svg_path, bbox_inches="tight", facecolor="white")

plt.show()

print(f"Saved PNG to: {png_path.resolve()}")
print(f"Saved SVG to: {svg_path.resolve()}")

In [ ]:
from pathlib import Path
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from PIL import Image, ImageChops

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter


# ============================================================
# Output files
# ============================================================
out_dir = Path(".")
png_path = out_dir / "fixed_vqc_noise_test_parity_accuracy_clean.png"
svg_path = out_dir / "fixed_vqc_noise_test_parity_accuracy_clean.svg"


# ============================================================
# Helper: trim white margins from image
# ============================================================
def trim_white_margin(image):
    image = image.convert("RGB")
    background = Image.new("RGB", image.size, (255, 255, 255))
    diff = ImageChops.difference(image, background)
    bbox = diff.getbbox()

    if bbox is None:
        return image

    return image.crop(bbox)


# ============================================================
# Qiskit helper
# ============================================================
def render_circuit_image(qc, dpi=900, fontsize=12, subfontsize=9):
    fig = qc.draw(
        output="mpl",
        fold=-1,
        idle_wires=False,
        style={
            "fontsize": fontsize,
            "subfontsize": subfontsize,
            "backgroundcolor": "#ffffff",
        },
    )

    buf = BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.03,
    )
    plt.close(fig)

    buf.seek(0)
    image = Image.open(buf)
    image = trim_white_margin(image)
    img_array = np.asarray(image)
    buf.close()

    return img_array


# ============================================================
# Build illustrative fixed trained VQC
# ============================================================
theta0 = Parameter("θ0*")
theta1 = Parameter("θ1*")
theta2 = Parameter("θ2*")
theta3 = Parameter("θ3*")

fixed_vqc = QuantumCircuit(2, name="Fixed VQC")
fixed_vqc.ry(theta0, 0)
fixed_vqc.ry(theta1, 1)
fixed_vqc.cx(0, 1)
fixed_vqc.ry(theta2, 0)
fixed_vqc.ry(theta3, 1)
fixed_vqc.measure_all()

vqc_img = render_circuit_image(
    fixed_vqc,
    dpi=900,
    fontsize=12,
    subfontsize=9,
)


# ============================================================
# Build illustrative test circuit
# ============================================================
x0 = Parameter("x0")
x1 = Parameter("x1")

test_circuit = QuantumCircuit(2, name="Test Circuit")

# Simple feature input stage
test_circuit.ry(x0, 0)
test_circuit.ry(x1, 1)

# Fixed trained VQC stage
test_circuit.ry(theta0, 0)
test_circuit.ry(theta1, 1)
test_circuit.cx(0, 1)
test_circuit.ry(theta2, 0)
test_circuit.ry(theta3, 1)

# Measurement for test prediction
test_circuit.measure_all()

test_circuit_img = render_circuit_image(
    test_circuit,
    dpi=900,
    fontsize=10,
    subfontsize=8,
)


# ============================================================
# Main figure
# ============================================================
fig, ax = plt.subplots(figsize=(18, 4.6), dpi=300)
ax.set_xlim(0, 18)
ax.set_ylim(0, 4.6)
ax.axis("off")


# ============================================================
# Drawing helpers
# ============================================================
def rounded_box(x, y, w, h, edgecolor, facecolor, linewidth=2.2):
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.045,rounding_size=0.12",
        linewidth=linewidth,
        edgecolor=edgecolor,
        facecolor=facecolor,
    )
    ax.add_patch(patch)
    return patch


def title(x, y, text, color, fontsize=14):
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold",
        color=color,
    )


def note(x, y, text, color, fontsize=8.5):
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        color=color,
    )


def arrow(x1, y1, x2, y2):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            arrowstyle="simple",
            mutation_scale=18,
            linewidth=0,
            color="#222222",
        )
    )


# ============================================================
# Layout
# ============================================================
box_y = 0.88
box_h = 2.70

w1 = 3.50
w2 = 2.45
w3 = 2.75
w4 = 2.75
w5 = 2.25

x1_box = 0.35
x2_box = 4.25
x3_box = 7.10
x4_box = 10.25
x5_box = 13.40

mid_y = box_y + box_h / 2


# ============================================================
# Box 1: Fixed Trained VQC
# ============================================================
rounded_box(
    x1_box,
    box_y,
    w1,
    box_h,
    edgecolor="#1d4ed8",
    facecolor="#eff6ff",
)

title(x1_box + w1 / 2, 3.25, "Fixed Trained VQC", "#1d4ed8")
note(x1_box + w1 / 2, 3.00, "reuse learned parameters", "#1e3a8a")

ax.imshow(
    vqc_img,
    extent=[x1_box + 0.28, x1_box + w1 - 0.28, 1.55, 2.65],
    aspect="auto",
    zorder=3,
)

note(x1_box + w1 / 2, 1.18, "no retraining", "#1e3a8a")


# ============================================================
# Box 2: Add Noise
# ============================================================
rounded_box(
    x2_box,
    box_y,
    w2,
    box_h,
    edgecolor="#b91c1c",
    facecolor="#fff7f7",
)

title(x2_box + w2 / 2, 3.25, "Add Noise", "#b91c1c")
note(x2_box + w2 / 2, 3.00, "simulated noise model", "#7f1d1d")

rng = np.random.default_rng(42)
x_noise = rng.uniform(x2_box + 0.50, x2_box + w2 - 0.50, 10)
y_noise = rng.uniform(1.62, 2.25, 10)

ax.scatter(
    x_noise,
    y_noise,
    s=30,
    color="#ef4444",
    alpha=0.70,
    edgecolors="none",
)

for x0, y0 in [(x2_box + 0.55, 1.85), (x2_box + 1.50, 1.85)]:
    xs = np.linspace(0, 0.38, 80) + x0
    ys = 0.05 * np.sin(np.linspace(0, 4 * np.pi, 80)) + y0
    ax.plot(xs, ys, color="#dc2626", linewidth=1.7)

note(x2_box + w2 / 2, 1.18, "depolarizing noise", "#7f1d1d")


# ============================================================
# Box 3: Run Test Circuits
# ============================================================
rounded_box(
    x3_box,
    box_y,
    w3,
    box_h,
    edgecolor="#7c3aed",
    facecolor="#f5f3ff",
)

title(x3_box + w3 / 2, 3.25, "Run Test Circuits", "#6d28d9")
note(x3_box + w3 / 2, 3.00, "evaluate test samples", "#4c1d95")

# Qiskit-rendered test circuit
ax.imshow(
    test_circuit_img,
    extent=[x3_box + 0.23, x3_box + w3 - 0.23, 1.50, 2.48],
    aspect="auto",
    zorder=3,
)

note(x3_box + w3 / 2, 1.23, "one circuit per test input", "#4c1d95")
note(x3_box + w3 / 2, 1.06, "repeated over test set", "#4c1d95", fontsize=7.8)


# ============================================================
# Box 4: Parity Vote
# ============================================================
rounded_box(
    x4_box,
    box_y,
    w4,
    box_h,
    edgecolor="#0369a1",
    facecolor="#f0f9ff",
)

title(x4_box + w4 / 2, 3.25, "Parity Vote", "#0369a1")
note(x4_box + w4 / 2, 3.00, "bitstrings become labels", "#075985")

row_y = 1.95
center_x = x4_box + w4 / 2

# Counts panel
counts_w = 0.86
counts_h = 0.86
counts_x = x4_box + 0.33
counts_y = row_y - counts_h / 2

ax.add_patch(
    FancyBboxPatch(
        (counts_x, counts_y),
        counts_w,
        counts_h,
        boxstyle="round,pad=0.025,rounding_size=0.07",
        linewidth=1.15,
        edgecolor="#0369a1",
        facecolor="white",
    )
)

ax.text(
    counts_x + counts_w / 2,
    counts_y + counts_h - 0.16,
    "Counts",
    ha="center",
    va="center",
    fontsize=7.8,
    fontweight="bold",
    color="#075985",
)

count_lines = ["00: 42", "01: 18", "10: 15", "11: 25"]

for i, line in enumerate(count_lines):
    ax.text(
        counts_x + 0.16,
        counts_y + counts_h - 0.34 - i * 0.145,
        line,
        ha="left",
        va="center",
        fontsize=6.9,
        color="#075985",
    )

# Label badge
label_w = 0.72
label_h = 0.44
label_x = x4_box + w4 - 0.33 - label_w
label_y = row_y - label_h / 2

ax.add_patch(
    FancyBboxPatch(
        (label_x, label_y),
        label_w,
        label_h,
        boxstyle="round,pad=0.025,rounding_size=0.08",
        linewidth=1.15,
        edgecolor="#0369a1",
        facecolor="#bae6fd",
    )
)

ax.text(
    label_x + label_w / 2,
    label_y + label_h / 2,
    "Label",
    ha="center",
    va="center",
    fontsize=8.3,
    fontweight="bold",
    color="#075985",
)

# Arrow between counts and label
ax.add_patch(
    FancyArrowPatch(
        (counts_x + counts_w + 0.14, row_y),
        (label_x - 0.12, row_y),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.35,
        color="#0369a1",
    )
)

ax.text(
    center_x,
    1.22,
    "even / odd majority",
    ha="center",
    va="center",
    fontsize=7.4,
    color="#075985",
)


# ============================================================
# Box 5: Accuracy
# ============================================================
rounded_box(
    x5_box,
    box_y,
    w5,
    box_h,
    edgecolor="#166534",
    facecolor="#f0fdf4",
)

title(x5_box + w5 / 2, 3.25, "Accuracy", "#166534")
note(x5_box + w5 / 2, 3.00, "compare with true labels", "#14532d")

base_x = x5_box + 0.62
base_y = 1.45
heights = [0.48, 0.86, 0.62]

ax.plot(
    [base_x - 0.10, base_x + 1.08],
    [base_y, base_y],
    color="#14532d",
    linewidth=1.2,
)

for i, h in enumerate(heights):
    ax.add_patch(
        Rectangle(
            (base_x + i * 0.30, base_y),
            0.17,
            h,
            linewidth=1.0,
            edgecolor="#166534",
            facecolor="#86efac",
        )
    )

note(x5_box + w5 / 2, 1.18, "classification score", "#14532d")


# ============================================================
# Arrows
# ============================================================
arrow(x1_box + w1 + 0.14, mid_y, x2_box - 0.14, mid_y)
arrow(x2_box + w2 + 0.14, mid_y, x3_box - 0.14, mid_y)
arrow(x3_box + w3 + 0.14, mid_y, x4_box - 0.14, mid_y)
arrow(x4_box + w4 + 0.14, mid_y, x5_box - 0.14, mid_y)


# ============================================================
# Caption
# ============================================================
ax.text(
    9.0,
    0.32,
    "The trained VQC is kept fixed, evaluated with simulated noise, converted to class predictions by parity vote, and scored by accuracy.",
    ha="center",
    va="center",
    fontsize=10,
    color="#374151",
)


# ============================================================
# Save and show
# ============================================================
fig.tight_layout()

fig.savefig(
    png_path,
    bbox_inches="tight",
    facecolor="white",
    dpi=600,
)

fig.savefig(
    svg_path,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

print(f"Saved PNG to: {png_path.resolve()}")
print(f"Saved SVG to: {svg_path.resolve()}")